# Expert Review Interface for Phishing Detection
## Phase 2: Data Annotation and Preprocessing

This notebook provides an interactive interface for domain experts to review and refine automated annotations.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
import sys
sys.path.append('../scripts')
from automated_tagging import AutomatedTagger
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime

## 1. Load Data and Initialize Tagger

In [ ]:
# Load dataset
data_path = Path('../data/raw/phishing_email.csv')
df = pd.read_csv(data_path, nrows=1000)  # Load subset for review
print(f"Loaded {len(df)} samples for review")

# Initialize automated tagger
tagger = AutomatedTagger()
print("Automated tagger initialized")

## 2. Generate Sample for Review

In [ ]:
# Select random samples for review
REVIEW_SAMPLE_SIZE = 100

# Stratified sampling to ensure balanced representation
phishing_samples = df[df['label'] == 1].sample(n=min(50, len(df[df['label'] == 1])))
legitimate_samples = df[df['label'] == 0].sample(n=min(50, len(df[df['label'] == 0])))

review_df = pd.concat([phishing_samples, legitimate_samples]).reset_index(drop=True)
print(f"Selected {len(review_df)} samples for expert review")
print(f"  - Phishing: {len(phishing_samples)}")
print(f"  - Legitimate: {len(legitimate_samples)}")

## 3. Apply Automated Tagging

In [ ]:
# Apply automated tagging to review samples
print("Applying automated tagging...")

review_df['auto_tags'] = ''
review_df['confidence'] = 0.0
review_df['risk_level'] = ''

for idx, row in review_df.iterrows():
    result = tagger.tag_email(row['text_combined'])
    review_df.at[idx, 'auto_tags'] = ', '.join(result['tags'])
    review_df.at[idx, 'confidence'] = result['confidence']
    review_df.at[idx, 'risk_level'] = result['risk_level']
    
print("✓ Automated tagging complete")

## 4. Expert Review Interface

In [ ]:
class ExpertReviewInterface:
    def __init__(self, df):
        self.df = df.copy()
        self.current_index = 0
        self.reviews = []
        self.expert_annotations = {}
        
        # Create UI components
        self.create_widgets()
        
    def create_widgets(self):
        # Navigation
        self.prev_btn = widgets.Button(description='Previous', disabled=True)
        self.next_btn = widgets.Button(description='Next')
        self.progress = widgets.IntProgress(value=0, min=0, max=len(self.df), description='Progress:')
        
        # Email display
        self.email_display = widgets.HTML(value='')
        
        # Automated tags display
        self.auto_tags_display = widgets.HTML(value='')
        
        # Expert review inputs
        self.correct_label = widgets.RadioButtons(
            options=['Phishing', 'Legitimate', 'Unsure'],
            description='Correct Label:',
            disabled=False
        )
        
        self.tag_checkboxes = widgets.SelectMultiple(
            options=['high_urgency', 'contains_threats', 'financial_content', 
                    'spelling_errors', 'suspicious_domain', 'ip_address_url',
                    'shortened_url', 'social_engineering', 'brand_impersonation'],
            description='Tags:',
            disabled=False
        )
        
        self.confidence_slider = widgets.FloatSlider(
            value=0.5,
            min=0,
            max=1.0,
            step=0.1,
            description='Confidence:',
            readout_format='.1%'
        )
        
        self.notes = widgets.Textarea(
            placeholder='Add notes about this sample...',
            description='Notes:',
            disabled=False
        )
        
        self.save_btn = widgets.Button(description='Save Review', button_style='success')
        
        # Button callbacks
        self.prev_btn.on_click(self.prev_sample)
        self.next_btn.on_click(self.next_sample)
        self.save_btn.on_click(self.save_review)
        
    def display_sample(self):
        """Display current sample for review."""
        row = self.df.iloc[self.current_index]
        
        # Update email display
        email_html = f"""
        <div style="border: 1px solid #ddd; padding: 10px; background: #f9f9f9;">
            <h4>Sample {self.current_index + 1} of {len(self.df)}</h4>
            <p><b>True Label:</b> {'Phishing' if row['label'] == 1 else 'Legitimate'}</p>
            <p><b>Email Content:</b></p>
            <div style="border: 1px solid #ccc; padding: 10px; background: white; max-height: 300px; overflow-y: auto;">
                {row['text_combined'][:1000]}{'...' if len(str(row['text_combined'])) > 1000 else ''}
            </div>
        </div>
        """
        self.email_display.value = email_html
        
        # Update automated tags display
        tags_html = f"""
        <div style="border: 1px solid #ddd; padding: 10px; background: #f0f8ff;">
            <h4>Automated Analysis</h4>
            <p><b>Tags:</b> {row['auto_tags'] if row['auto_tags'] else 'None'}</p>
            <p><b>Confidence:</b> {row['confidence']:.1%}</p>
            <p><b>Risk Level:</b> <span style="color: {'red' if row['risk_level'] == 'high' else 'orange' if row['risk_level'] == 'medium' else 'green'}">{row['risk_level'].upper()}</span></p>
        </div>
        """
        self.auto_tags_display.value = tags_html
        
        # Update progress
        self.progress.value = self.current_index + 1
        
        # Update navigation buttons
        self.prev_btn.disabled = self.current_index == 0
        self.next_btn.disabled = self.current_index == len(self.df) - 1
        
        # Load previous review if exists
        if self.current_index in self.expert_annotations:
            ann = self.expert_annotations[self.current_index]
            self.correct_label.value = ann['label']
            self.tag_checkboxes.value = ann['tags']
            self.confidence_slider.value = ann['confidence']
            self.notes.value = ann['notes']
        else:
            # Set defaults based on true label
            self.correct_label.value = 'Phishing' if row['label'] == 1 else 'Legitimate'
            self.tag_checkboxes.value = []
            self.confidence_slider.value = row['confidence']
            self.notes.value = ''
    
    def save_review(self, b):
        """Save expert review for current sample."""
        self.expert_annotations[self.current_index] = {
            'sample_id': self.current_index,
            'label': self.correct_label.value,
            'tags': list(self.tag_checkboxes.value),
            'confidence': self.confidence_slider.value,
            'notes': self.notes.value,
            'timestamp': datetime.now().isoformat()
        }
        
        # Show confirmation
        print(f"✓ Review saved for sample {self.current_index + 1}")
        
        # Auto-advance to next sample
        if self.current_index < len(self.df) - 1:
            self.next_sample(None)
    
    def prev_sample(self, b):
        """Navigate to previous sample."""
        if self.current_index > 0:
            self.current_index -= 1
            self.display_sample()
    
    def next_sample(self, b):
        """Navigate to next sample."""
        if self.current_index < len(self.df) - 1:
            self.current_index += 1
            self.display_sample()
    
    def show(self):
        """Display the review interface."""
        # Layout
        navigation = widgets.HBox([self.prev_btn, self.next_btn, self.progress])
        review_form = widgets.VBox([
            self.correct_label,
            self.tag_checkboxes,
            self.confidence_slider,
            self.notes,
            self.save_btn
        ])
        
        interface = widgets.VBox([
            navigation,
            self.email_display,
            self.auto_tags_display,
            review_form
        ])
        
        # Display first sample
        self.display_sample()
        
        return interface
    
    def export_reviews(self):
        """Export expert reviews to file."""
        output_path = Path('../data/processed/expert_reviews.json')
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w') as f:
            json.dump(self.expert_annotations, f, indent=2)
        
        print(f"✓ Expert reviews exported to {output_path}")
        return self.expert_annotations

## 5. Launch Review Interface

In [ ]:
# Create and display the review interface
reviewer = ExpertReviewInterface(review_df)
reviewer.show()

## 6. Export Reviews and Calculate Agreement

In [ ]:
# Export reviews
reviews = reviewer.export_reviews()

# Calculate agreement statistics
if reviews:
    print("\n" + "="*50)
    print("REVIEW STATISTICS")
    print("="*50)
    
    total_reviewed = len(reviews)
    print(f"Total samples reviewed: {total_reviewed}")
    
    # Calculate label agreement
    label_agreement = 0
    for idx, review in reviews.items():
        true_label = 'Phishing' if review_df.iloc[int(idx)]['label'] == 1 else 'Legitimate'
        if review['label'] == true_label:
            label_agreement += 1
    
    print(f"Label agreement with ground truth: {label_agreement}/{total_reviewed} ({100*label_agreement/total_reviewed:.1f}%)")
    
    # Tag statistics
    all_tags = []
    for review in reviews.values():
        all_tags.extend(review['tags'])
    
    from collections import Counter
    tag_counts = Counter(all_tags)
    
    print("\nMost common tags:")
    for tag, count in tag_counts.most_common(5):
        print(f"  - {tag}: {count} occurrences")
    
    # Confidence statistics
    confidences = [review['confidence'] for review in reviews.values()]
    print(f"\nAverage confidence: {np.mean(confidences):.1%}")
    print(f"Confidence range: {min(confidences):.1%} - {max(confidences):.1%}")

## 7. Save Enhanced Dataset

In [ ]:
# Merge expert reviews back into dataset
if reviews:
    for idx, review in reviews.items():
        idx = int(idx)
        review_df.at[idx, 'expert_label'] = review['label']
        review_df.at[idx, 'expert_tags'] = ', '.join(review['tags'])
        review_df.at[idx, 'expert_confidence'] = review['confidence']
        review_df.at[idx, 'expert_notes'] = review['notes']
    
    # Save enhanced dataset
    output_path = Path('../data/processed/expert_reviewed_samples.csv')
    review_df.to_csv(output_path, index=False)
    print(f"\n✓ Enhanced dataset saved to {output_path}")